# PARTE 02 - Chatbot RAG alimentado pela planilha na camada Gold

## Imports

In [ ]:
!pip install langchain langchain-google-genai gspread --quiet

In [ ]:
import gspread
from google.colab import auth
from google.auth import default
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings

## Leitura da Planilha

In [ ]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

PLANILHA_URL = 'url_da_planilha'

planilha = gc.open_by_url(PLANILHA_URL)
aba_gold = planilha.worksheet('gold')

dados = aba_gold.get_all_records()
print(f"Filmes carregados: {len(dados)}")

for filme in dados:
    print(f"- {filme['titulo']} ({filme['ano']}) — {filme['avaliacao']}/100")

## Lista com filmes

In [ ]:
documentos = []
for filme in dados:
    texto = f"""
      Título: {filme['titulo']}
      Ano: {filme['ano']}
      Avaliação IMDB: {filme['avaliacao']}/100
      Resumo: {filme['resumo_rag']}
      """.strip()

    documentos.append(texto)

print(f"Documentos criados: {len(documentos)}")
print(f"\nExemplo:")
print(documentos[0])
documentos

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

vectorstore = InMemoryVectorStore.from_texts(
    documentos,
    embedding=embeddings)

print(f"Embeddings gerados e armazenados no InMemoryVectorStore.")
print(f"Total de documentos indexados: {len(documentos)}")

In [ ]:
pergunta = "matrix"
vectorstore.similarity_search(pergunta, k=1)

## Inicializando o Modelo

In [ ]:
llm = init_chat_model(
    model="gemini-2.5-flash",
    model_provider="google_genai",
    api_key=GOOGLE_API_KEY
)

print("Modelo inicializado.")


In [ ]:
print("=" * 50)
print("IMDB Chatbot — Powered by RAG")
print("Digite 'sair' para encerrar")
print("=" * 50)

while True:
    pergunta = input("\nVocê: ").strip()

    if pergunta.lower() == 'sair':
        print("Encerrando o chatbot.")
        break

    if not pergunta:
        continue

    # Busca semântica
    docs_relevantes = vectorstore.similarity_search(pergunta, k=3)

    # Monta o contexto
    contexto = "\n\n".join([doc.page_content for doc in docs_relevantes])

    # Gera a resposta
    resposta = llm.invoke(f"""
Você é um assistente especializado em filmes do IMDB.
Use apenas as informações abaixo para responder a pergunta do usuário.
Se a informação não estiver disponível, diga que não tem essa informação.
Responda sempre em português de forma amigável e informativa.

Contexto:
{contexto}

Pergunta: {pergunta}
""")

    print(f"\nChatbot: {resposta.content}")